## Introduction

This notebook retrieves the web version of Benefits product newsletters from HubSpot's archive and stores them for later publication on the Benefits docs site.

### Setup

Requires a HubSpot [legacy private app](https://developers.hubspot.com/docs/apps/legacy-apps/private-apps/overview) with the `content` scope.

The app will provide an access token that should be stored in an environment variable called `HUBSPOT_ACCESS_TOKEN`.

You can copy the sample environment file to get started; run the following command from the root of this repository:

```bash
cp .env.sample .env
```

Then open `.env` and fill in with your access token.

**The block below should be run before attempting any other blocks in this notebook.**
It will need to be re-run after restarting the kernel, as well.

In [ ]:
import os
import urllib
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from hubspot import HubSpot

from data.utils import hubspot_get_all_pages, hubspot_to_df


ACCESS_TOKEN = os.environ["HUBSPOT_ACCESS_TOKEN"]
PAGE_SIZE = int(os.environ.get("HUBSPOT_PAGE_SIZE", 10))

hubspot = HubSpot(access_token=ACCESS_TOKEN)

## Export

In [ ]:
# retrieve all emails
emails_response = hubspot_get_all_pages(
    hubspot.marketing.emails.marketing_emails_api,
    page_size=PAGE_SIZE,
    included_properties=["publishDate", "subject", "subscriptionDetails", "webversion"],
    sort=["updated_at"],
)
emails_df = hubspot_to_df(emails_response)

# filter down to emails sent to Benefits Product Updates subscribers
emails_df_benefits = emails_df[emails_df["subscription_details.subscription_id"].eq("313573410")]

# loop through emails
for row_index, row in emails_df_benefits.iterrows():
    # scrape HTML
    r = requests.get(row['webversion.url'])
    soup = BeautifulSoup(r.content, "html.parser")

    # collect images and store in shared images folder (overwriting any with same filename)
    for img in soup.find_all("img"):
        img_src = img["src"]
        img_src_parsed = urllib.parse.urlsplit(img_src)
        filename = os.path.basename(img_src_parsed.path)

        # download image if we haven't yet
        decoded_filename = urllib.parse.unquote(filename)
        download_path = f"benefits_emails/images/{decoded_filename}"
        if not Path(download_path).exists():
            print(f"Downloading {decoded_filename} from {img_src_parsed._replace(query='').geturl()}")
            with open(download_path, mode="wb") as file:
                img_r = requests.get(img_src_parsed._replace(query="").geturl())  # drop query params to get largest size
                file.write(img_r.content)

        # replace original src with relative path and drop responsive attrs
        img["src"] = f"images/{filename}"
        del img["sizes"]
        del img["srcset"]

    # write the updated HTML to a file named with the sending date
    send_time = row["publish_date"].tz_convert("America/Los_Angeles")
    with open(f"benefits_emails/{send_time.strftime('%Y-%m-%d')}.html", "w") as file:
        file.write(str(soup))
